# 🏦 Financial Fraud Detection Pipeline
## Notebook 3: Modeling, Evaluation & Export

**Goal:** Train Random Forest and XGBoost classifiers, evaluate with fraud-appropriate metrics, compare models, and export predictions for Tableau.

**Why not just use accuracy?**  
With 0.17% fraud, a model predicting 'legitimate' every time gets 99.83% accuracy — but catches zero fraud.  
We use **Precision, Recall, F1, ROC-AUC, and PR-AUC** instead.

| Metric | What it tells us |
|---|---|
| **Recall** | Of all actual frauds, how many did we catch? (most important) |
| **Precision** | Of all fraud predictions, how many were actually fraud? |
| **F1** | Balance between precision and recall |
| **ROC-AUC** | Overall discrimination ability |
| **PR-AUC** | Performance on imbalanced data (more informative than ROC) |

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
import time

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded ✅')

## 2. Load Preprocessed Data

In [ ]:
train = pd.read_csv('../data/train_resampled.csv')
test  = pd.read_csv('../data/test.csv')

X_train = train.drop(columns=['Class'])
y_train = train['Class']
X_test  = test.drop(columns=['Class'])
y_test  = test['Class']

print(f'Training set: {X_train.shape} | Fraud: {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'Test set:     {X_test.shape}  | Fraud: {y_test.sum():,} ({y_test.mean()*100:.4f}%)')
print('Data loaded ✅')

## 3. Train Models

In [ ]:
# --- Random Forest ---
print('Training Random Forest...')
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_time = time.time() - t0
print(f'Random Forest trained in {rf_time:.1f}s ✅')

# --- XGBoost ---
print('\nTraining XGBoost...')
t0 = time.time()
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)
xgb_time = time.time() - t0
print(f'XGBoost trained in {xgb_time:.1f}s ✅')

## 4. Generate Predictions & Probabilities

In [ ]:
# Predictions
rf_pred  = rf.predict(X_test)
xgb_pred = xgb.predict(X_test)

# Probabilities (for ROC / PR curves)
rf_proba  = rf.predict_proba(X_test)[:, 1]
xgb_proba = xgb.predict_proba(X_test)[:, 1]

print('Predictions generated ✅')

## 5. Classification Reports

In [ ]:
for name, pred in [('Random Forest', rf_pred), ('XGBoost', xgb_pred)]:
    print(f'{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(classification_report(y_test, pred, target_names=['Legitimate', 'Fraud']))
    print()

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, pred, name in zip(axes,
                           [rf_pred, xgb_pred],
                           ['Random Forest', 'XGBoost']):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt=',', cmap='Blues', ax=ax,
                xticklabels=['Pred: Legit', 'Pred: Fraud'],
                yticklabels=['Actual: Legit', 'Actual: Fraud'],
                linewidths=0.5, cbar=False)
    ax.set_title(f'{name}\nConfusion Matrix', fontsize=12, fontweight='bold')

    tn, fp, fn, tp = cm.ravel()
    ax.set_xlabel(f'True Positives (Fraud Caught): {tp} | False Negatives (Missed): {fn}', fontsize=9)

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/confusion_matrices.png ✅')

## 7. ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))

for proba, name, color in [(rf_proba, 'Random Forest', '#4C72B0'),
                            (xgb_proba, 'XGBoost', '#DD8452')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline')
plt.xlabel('False Positive Rate', fontsize=11)
plt.ylabel('True Positive Rate', fontsize=11)
plt.title('ROC Curve Comparison', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/roc_curves.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/roc_curves.png ✅')

## 8. Precision-Recall Curves

PR curves are more informative than ROC when classes are highly imbalanced.

In [ ]:
plt.figure(figsize=(8, 6))

for proba, name, color in [(rf_proba, 'Random Forest', '#4C72B0'),
                            (xgb_proba, 'XGBoost', '#DD8452')]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    plt.plot(rec, prec, label=f'{name} (AP = {ap:.4f})', color=color, linewidth=2)

baseline = y_test.mean()
plt.axhline(baseline, color='gray', linestyle='--', linewidth=1,
            label=f'Baseline (fraud rate = {baseline:.4f})')
plt.xlabel('Recall', fontsize=11)
plt.ylabel('Precision', fontsize=11)
plt.title('Precision-Recall Curve Comparison', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/pr_curves.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/pr_curves.png ✅')

## 9. Feature Importance (XGBoost)

In [ ]:
feat_imp = pd.Series(xgb.feature_importances_, index=X_train.columns)
top15 = feat_imp.sort_values(ascending=True).tail(15)

plt.figure(figsize=(10, 6))
bars = plt.barh(top15.index, top15.values, color='#4C72B0', edgecolor='white')
plt.title('XGBoost — Top 15 Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
for bar, val in zip(bars, top15.values):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/feature_importance.png', bbox_inches='tight')
plt.show()
print('Saved to outputs/feature_importance.png ✅')

## 10. Model Comparison Summary

In [ ]:
results = []
for name, pred, proba in [('Random Forest', rf_pred, rf_proba),
                            ('XGBoost',       xgb_pred, xgb_proba)]:
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    results.append({
        'Model':       name,
        'Precision':   round(precision_score(y_test, pred), 4),
        'Recall':      round(recall_score(y_test, pred), 4),
        'F1':          round(f1_score(y_test, pred), 4),
        'ROC-AUC':     round(roc_auc_score(y_test, proba), 4),
        'PR-AUC':      round(average_precision_score(y_test, proba), 4),
        'Fraud Caught (TP)':   tp,
        'Fraud Missed (FN)':   fn,
        'False Alarms (FP)':   fp,
    })

results_df = pd.DataFrame(results).set_index('Model')
print('=== Model Comparison ===')
display(results_df.style.highlight_max(axis=0, color='#c6efce').highlight_min(axis=0, color='#ffc7ce'))

## 11. Export Predictions for Tableau

In [ ]:
# Use best model (XGBoost) for export
# Re-run on full original test set for Tableau
export_df = test.copy()
export_df['RF_Prediction']    = rf_pred
export_df['XGB_Prediction']   = xgb_pred
export_df['RF_Fraud_Prob']    = rf_proba
export_df['XGB_Fraud_Prob']   = xgb_proba
export_df['RF_Label']         = rf_pred.map({0: 'Legitimate', 1: 'Fraud'})
export_df['XGB_Label']        = xgb_pred.map({0: 'Legitimate', 1: 'Fraud'})
export_df['Actual_Label']     = y_test.map({0: 'Legitimate', 1: 'Fraud'}).values

# Outcome column for Tableau filtering
def outcome(row):
    if row['Class'] == 1 and row['XGB_Prediction'] == 1: return 'True Positive'
    if row['Class'] == 0 and row['XGB_Prediction'] == 0: return 'True Negative'
    if row['Class'] == 0 and row['XGB_Prediction'] == 1: return 'False Positive'
    if row['Class'] == 1 and row['XGB_Prediction'] == 0: return 'False Negative'

export_df['XGB_Outcome'] = export_df.apply(outcome, axis=1)

export_df.to_csv('../outputs/predictions.csv', index=False)

print(f'Exported {len(export_df):,} rows to outputs/predictions.csv ✅')
print('\nOutcome breakdown (XGBoost):')
print(export_df['XGB_Outcome'].value_counts().to_string())
print('\nColumns exported:')
print(export_df.columns.tolist())

## 12. Final Summary

| | Random Forest | XGBoost |
|---|---|---|
| **Best for** | Interpretability, fast training | Highest accuracy, handles imbalance natively |
| **Key metric** | Recall — catching fraud matters most | Recall — catching fraud matters most |
| **Output** | `predictions.csv` with probabilities | Ready for Tableau dashboard |

### ➡️ Next Step: Open Tableau and connect to `outputs/predictions.csv` to build the fraud detection dashboard